# Day 1 | Capstone Assignment: The VIP Recommendation Sandbox
**Author: Sattaya Singkul**

Welcome to the Day 1 Final Assignment! You are now the lead engineer at **"Aether Electronics"**, a boutique shop for high-end gear.

**Objective**: Your goal is to build a high-performance recommendation engine from scratch using RedisVL. 
**Rules**:
1. You must achieve the exact results specified in the **Verifier Cells**.
2. If the verifier fails (`AssertionError`), your implementation is incorrect.

Good luck!


In [ ]:
!pip install redisvl pandas sentence_transformers

### Task 0: Establishing Connection
Connect to your `redis-vector-db` container and initialize the `SearchIndex` client.


In [1]:
# YOUR CODE HERE
redis_url = "redis://redis-vector-db:6379"
# ... (Establish connection)


### Task 1: The Schema Architect
Define an `IndexSchema` named `vip_products`. 
It must contain:
1. `brand`: Tag
2. `price`: Numeric
3. `description`: Text
4. `embedding`: Vector (Dims: 384, Metric: Cosine, Algorithm: HNSW)


In [4]:
# YOUR CODE HERE (Define schema and create index)
from redisvl.schema import IndexSchema

schema_dict = {
    "index": {
        "name": "vip_products",
        "prefix": "item:", 
        "storage_type": "json"  # <--- Change this from 'hash' (or default) to 'json'
    },
    "fields": [
        {"name": "product_id", "type": "tag"},       # Exact match queries
        {"name": "category", "type": "tag"},         # Filtering
        {"name": "brand", "type": "tag"},            # Filtering
        {"name": "price", "type": "numeric"},        # Range filtering
        {"name": "description", "type": "text"},     # Full text search fallback
        {"name": "location", "type": "geo"},         # <--- NEW: Geo-Spatial field for local discovery
        {
            "name": "embedding",
            "type": "vector",
            "attrs": {
                "dims": 384,                     # Dimension count matching our HuggingFace model
                "distance_metric": "cosine",     # Best for semantic NLP descriptions
                "algorithm": "hnsw",             # HNSW Graph!
                "datatype": "float32",
                "m": 16,                         # Standard HNSW link density
                "ef_construction": 200
            }
        }
    ]
}

schema = {
    "index": {
        "name": "vip_products",
        "prefix": "product",
        "storage_type": "json"  # <--- Change this from 'hash' (or default) to 'json'
    },
    "fields": [
        # ... your other fields ...
        {
            "name": "embedding", 
            "type": "vector", 
            "attrs": {
                "dims": 384, # all-MiniLM-L6-v2 uses 384 dimensions
                "distance_metric": "cosine", 
                "algorithm": "flat"
            }
        }
    ]
}

from redisvl.index import SearchIndex
index = SearchIndex.from_dict(schema)

schema = IndexSchema.from_dict(schema_dict)

index = SearchIndex(schema, redis_url=redis_url)
index.create(overwrite=True) 

# VERIFIER (Do not modify)
assert index.name == "vip_products"
assert "description" in index.schema.field_names
print("✅ Task 1 Passed!")

✅ Task 1 Passed!


In [12]:
index.schema.fields

'ecommerce_catalog'

In [5]:
# # delete index data + value data in DB
# index.delete(drop=True)

In [3]:
# delete all data in DB
client = index.client
client.flushall()

True

### Task 2: Data Loading
Load the provided dataset into your index.


In [9]:
from sentence_transformers import SentenceTransformer
import time
embedder = SentenceTransformer('all-MiniLM-L6-v2')

raw_data = [
    {"brand": "Sony", "price": 400.0, "description": "Flagship noise cancelling headphones"},
    {"brand": "Sony", "price": 150.0, "description": "Budget wireless earbuds"},
    {"brand": "Apple", "price": 1200.0, "description": "High-end laptop for pros"},
    {"brand": "Apple", "price": 250.0, "description": "Premium wireless audio"},
    {"brand": "Logitech", "price": 80.0, "description": "Professional gaming mouse"},
]

# YOUR CODE HERE (Embed and Load)
docs_to_load = []
for p in raw_data:
    vector = embedder.encode(p["description"]).tolist() # convert numpy to byte for Redis
    # vector = embedder.encode(p["description"]).astype(np.float32).tobytes()
    doc = p.copy()
    doc["embedding"] = vector
    docs_to_load.append(doc)

start = time.time()
keys = index.load(docs_to_load)
dur = time.time() - start

print(f"Loaded {len(keys)} document vectors into Redis in {dur:.3f} seconds.")

# VERIFIER (Do not modify)
import time
time.sleep(1) # Allow for indexing
assert index.info()['num_docs'] == 5
print("✅ Task 2 Passed!")


✅ Task 2 Passed!


In [8]:
index.info()['num_docs']

5

### Task 3: The Hybrid Filter Challenge
Write a **Hybrid Search** that finds "Professional Audio" but **strictly** filters for `brand == 'Sony'` AND `price > 300`.


In [ ]:
# YOUR CODE HERE (vq = VectorQuery(...))

# VERIFIER (Do not modify)
results = index.query(vq)
assert len(results) == 1, f"Expected 1 hit, got {len(results)}"
assert results[0]['brand'] == 'Sony'
assert float(results[0]['price']) > 300
print("✅ Task 3 Passed!")


### Task 4: The Discovery Challenge (Radius Search)
Your customer likes the "High-end laptop" but wants to see **alternatives that are NOT Apple**.
Use `RangeQuery` with a `distance_threshold=0.25` and a filter to exclude `brand == 'Apple'`.


In [ ]:
# YOUR CODE HERE (rq = RangeQuery(...))

# VERIFIER (Do not modify)
results = index.query(rq)
for r in results:
    assert r['brand'] != 'Apple', f"Found Apple product: {r['description']}"
print("✅ Task 4 Passed!")


### Task 5: The Optimization Sprint (Semantic Cache)
Setup a `SemanticCache` with a `distance_threshold=0.1`. Store one entry and prove that a semantically similar query results in a cache **HIT**.


In [ ]:
# YOUR CODE HERE (cache = SemanticCache(...))

# VERIFIER (Do not modify)
# (Hit the cache with query: 'cost of pro laptop')
assert hit, "Cache missed! Ensure your threshold is correct and your entry is stored."
print("✅ Task 5 Passed!")
